# Geração de dados mockados — localização e status de motoristas

Objetivo: gerar ~10 registros por driver (10 drivers, ~100 linhas) para
exercitar point-in-time joins no historical features do Feast, e escrever
o resultado como uma Delta table local (offline store).

Dados 100% sintéticos, sem qualquer relação com clientes reais.

In [ ]:
from datetime import datetime, timedelta, timezone
from pathlib import Path

import numpy as np
import pandas as pd
from deltalake import DeltaTable, write_deltalake

## Parâmetros

In [ ]:
N_DRIVERS = 10
N_EVENTS_PER_DRIVER = 10
BASE_TIME = datetime.now(timezone.utc) - timedelta(days=10)
STATUS_CHOICES = ["available", "en_route", "busy", "offline"]
BASE_LAT, BASE_LON = -23.5505, -46.6333  # São Paulo, apenas referência de mock

try:
    NOTEBOOK_DIR = Path(__file__).resolve().parent
except NameError:
    NOTEBOOK_DIR = Path.cwd()
REPO_ROOT = NOTEBOOK_DIR.parent
DELTA_PATH = REPO_ROOT / "data" / "offline_store" / "driver_stats"

## Geração dos eventos por driver

Timestamps espaçados de forma não uniforme ao longo de 10 dias, para dar
margem a timestamps "as of" que caem entre eventos no historical features.
`created_timestamp` simula chegada tardia do dado (alguns segundos depois
do `event_timestamp`), usado para desduplicação no point-in-time join.

In [ ]:
rng = np.random.default_rng(seed=42)
rows = []
for driver_id in range(1000, 1000 + N_DRIVERS):
    t = BASE_TIME
    lat = BASE_LAT + rng.normal(0, 0.05)
    lon = BASE_LON + rng.normal(0, 0.05)
    for _ in range(N_EVENTS_PER_DRIVER):
        t = t + timedelta(minutes=int(rng.integers(30, 600)))
        lat += rng.normal(0, 0.01)
        lon += rng.normal(0, 0.01)
        event_ts = t
        created_ts = event_ts + timedelta(seconds=int(rng.integers(1, 120)))
        rows.append(
            {
                "driver_id": driver_id,
                "event_timestamp": event_ts,
                "created_timestamp": created_ts,
                "latitude": float(lat),
                "longitude": float(lon),
                "status": str(rng.choice(STATUS_CHOICES)),
            }
        )

df = pd.DataFrame(rows)
df["driver_id"] = df["driver_id"].astype("int64")
df["latitude"] = df["latitude"].astype("float32")
df["longitude"] = df["longitude"].astype("float32")
df["event_timestamp"] = pd.to_datetime(df["event_timestamp"], utc=True)
df["created_timestamp"] = pd.to_datetime(df["created_timestamp"], utc=True)

## Validação básica antes de escrever

In [ ]:
assert df.shape[0] == N_DRIVERS * N_EVENTS_PER_DRIVER
assert df["driver_id"].nunique() == N_DRIVERS
assert set(df["status"].unique()) <= set(STATUS_CHOICES)
print(df.shape)
df.head()

## Escrita como Delta table local

In [ ]:
DELTA_PATH.parent.mkdir(parents=True, exist_ok=True)
write_deltalake(str(DELTA_PATH), df, mode="overwrite")
print(f"Delta table escrita em: {DELTA_PATH}")

## Validação de releitura

In [ ]:
dt = DeltaTable(str(DELTA_PATH))
df_check = dt.to_pandas()
print(df_check.shape, df_check["driver_id"].nunique())
df_check.sort_values(["driver_id", "event_timestamp"]).head(10)